In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.metrics import f1_score, classification_report, hamming_loss, accuracy_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.multioutput import ClassifierChain
import mlflow
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 0. SETUP MLflow
# ==========================================
root_path = Path.cwd().parent
mlflow.set_tracking_uri((root_path / "mlruns").as_uri())
mlflow.set_experiment("08b_Threshold_Optimization")

# ==========================================
# 1. LOAD DATA
# ==========================================
print("⏳ 1. Memuat Data...")

train_smote = pd.read_csv(root_path / "Data/processed/train_selected_features.csv")
train_ori   = pd.read_csv(root_path / "Data/split/train_data.csv")
test_df     = pd.read_csv(root_path / "Data/split/test_data.csv")

selected_features = [
    'TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10',
    'VCL1', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10', 'VCL12', 'VCL13',
    'VCL14', 'VCL15', 'gender', 'engnat', 'age', 'religion', 'orientation',
    'race', 'voted', 'married', 'familysize'
]
target_cols = ['risk_depression', 'risk_anxiety', 'risk_stress']

# Data SMOTE untuk training final model
X_full_train_smote = train_smote[selected_features]
Y_full_train_smote = train_smote[target_cols].astype(int)

# Data ASLI untuk kalibrasi threshold
X_full_train_ori = train_ori[selected_features]
Y_full_train_ori = train_ori[target_cols].astype(int)

# Data test
X_test = test_df[selected_features]
Y_test = test_df[target_cols].astype(int)

print(f"✓ Train SMOTE  : {X_full_train_smote.shape}")
print(f"✓ Train Asli   : {X_full_train_ori.shape}")
print(f"✓ Test         : {X_test.shape}")

# ==========================================
# 2. SPLIT DATA ASLI → TRAIN_SUB + VALIDATION
# ==========================================
print("\n🔀 2. Split Data Asli → Train_sub & Validation (80-20)")

X_train_sub, X_val, Y_train_sub, Y_val = train_test_split(
    X_full_train_ori, Y_full_train_ori,
    test_size=0.2,
    random_state=42
)

print(f"✓ Train sub (asli) : {X_train_sub.shape}")
print(f"✓ Validation (asli): {X_val.shape}")

# Verifikasi distribusi
print(f"\nDistribusi Validation (asli):")
print(Y_val.mean().round(4))
print(f"\nDistribusi Test:")
print(Y_test.mean().round(4))

# ==========================================
# 3. LOAD FINAL MODEL DARI PICKLE
# ==========================================
print("\n📦 3. Load Final Model dari Pickle...")

model_path = root_path / "models/multilabel_xgboost_classifier_chain.pkl"
with open(model_path, "rb") as f:
    final_model = pickle.load(f)

# Ambil best params dari pickle untuk model sementara
sample_est  = final_model.estimators_[0]
best_params = {
    'n_estimators'   : getattr(sample_est, 'n_estimators',     300),
    'max_depth'      : getattr(sample_est, 'max_depth',        3),
    'learning_rate'  : getattr(sample_est, 'learning_rate',    0.05),
    'subsample'      : getattr(sample_est, 'subsample',        0.8),
    'colsample_bytree': getattr(sample_est, 'colsample_bytree', 0.7),
    'gamma'          : getattr(sample_est, 'gamma',            0.2),
    'min_child_weight': getattr(sample_est, 'min_child_weight', 1),
}

print(f"✓ Final model loaded: {type(final_model).__name__}")
print(f"✓ Best params dari pickle:")
for k, v in best_params.items():
    print(f"   - {k}: {v}")

# ==========================================
# 4. HELPER FUNCTIONS
# ==========================================
def chain_predict_proba(model, X):
    """
    Rekonstruksi manual predict_proba untuk ClassifierChain.
    Digunakan karena XGBoost 2.1.1 tidak kompatibel dengan
    ClassifierChain.predict_proba() dari sklearn.
    """
    X_arr      = X.values if hasattr(X, 'values') else X.copy()
    n_samples  = X_arr.shape[0]
    n_targets  = len(model.estimators_)
    all_probas = np.zeros((n_samples, n_targets))

    X_aug = X_arr.copy()
    for i, estimator in enumerate(model.estimators_):
        proba            = estimator.predict_proba(X_aug)[:, 1]
        all_probas[:, i] = proba
        pred_label       = (proba >= 0.5).astype(int).reshape(-1, 1)
        X_aug            = np.hstack([X_aug, pred_label])

    return [np.column_stack([1 - all_probas[:, i], all_probas[:, i]])
            for i in range(n_targets)]


def chain_predict_proba_threshold(model, X, thresholds):
    """
    Rekonstruksi predict_proba dengan threshold optimal.
    Augmentasi chain menggunakan threshold optimal bukan 0.5.
    """
    X_arr      = X.values if hasattr(X, 'values') else X.copy()
    n_samples  = X_arr.shape[0]
    n_targets  = len(model.estimators_)
    all_probas = np.zeros((n_samples, n_targets))

    X_aug = X_arr.copy()
    for i, estimator in enumerate(model.estimators_):
        proba            = estimator.predict_proba(X_aug)[:, 1]
        all_probas[:, i] = proba
        pred_label       = (proba >= thresholds[i]).astype(int).reshape(-1, 1)
        X_aug            = np.hstack([X_aug, pred_label])

    return [np.column_stack([1 - all_probas[:, i], all_probas[:, i]])
            for i in range(n_targets)]

print("\n✓ Helper functions defined")

with mlflow.start_run(run_name="XGBoost_ClassifierChain_Threshold_Optimization"):

    # ==========================================
    # 5. TRAINING MODEL SEMENTARA DI DATA ASLI
    #    (Hanya untuk kalibrasi threshold)
    # ==========================================
    print("\n🔧 5. Training model sementara di data asli (untuk kalibrasi threshold)...")

    base_xgb_temp = XGBClassifier(
        objective        = 'binary:logistic',
        eval_metric      = 'logloss',
        n_estimators     = best_params['n_estimators'],
        max_depth        = best_params['max_depth'],
        learning_rate    = best_params['learning_rate'],
        subsample        = best_params['subsample'],
        colsample_bytree = best_params['colsample_bytree'],
        gamma            = best_params['gamma'],
        min_child_weight = best_params['min_child_weight'],
        random_state     = 42,
        n_jobs           = -1
    )

    model_for_threshold = ClassifierChain(base_xgb_temp, order='random', random_state=42)
    model_for_threshold.fit(X_train_sub, Y_train_sub)
    print("✓ Model sementara selesai dilatih")
    print("  (Model ini hanya untuk kalibrasi threshold, akan dibuang setelahnya)")

    # ==========================================
    # 6. CARI THRESHOLD OPTIMAL DI VALIDATION ASLI
    # ==========================================
    print("\n🔍 6. Mencari Threshold Optimal di Validation ASLI...")

    y_val_proba = chain_predict_proba(model_for_threshold, X_val)

    best_thresholds      = []
    validation_f1_scores = {}

    print(f"\n   {'Target':<22} {'Threshold':>10} {'F1 Val':>10}")
    print(f"   {'-'*45}")

    for i, target_name in enumerate(target_cols):
        y_true    = Y_val.iloc[:, i]
        y_prob    = y_val_proba[i][:, 1]
        thresholds = np.arange(0.1, 0.9, 0.01)
        f1_scores  = []

        for t in thresholds:
            y_pred_temp = (y_prob >= t).astype(int)
            f1_scores.append(f1_score(y_true, y_pred_temp, zero_division=0))

        best_idx = np.argmax(f1_scores)
        best_t   = thresholds[best_idx]
        best_f1  = f1_scores[best_idx]

        best_thresholds.append(best_t)
        validation_f1_scores[target_name] = best_f1

        print(f"   {target_name:<22} {best_t:>10.2f} {best_f1:>10.4f}")

        mlflow.log_param(f"threshold_{target_name}", round(best_t, 2))
        mlflow.log_metric(f"validation_f1_{target_name}", best_f1)

    # Buang model sementara
    del model_for_threshold, base_xgb_temp
    print("\n✓ Model sementara dibuang")
    print(f"✓ Threshold hasil kalibrasi: {dict(zip(target_cols, [round(t,2) for t in best_thresholds]))}")

    # ==========================================
    # 7. RETRAIN FINAL MODEL DI FULL TRAIN SMOTE
    # ==========================================
    print("\n🔁 7. Retrain FINAL MODEL di Full Train SMOTE...")

    final_model.fit(X_full_train_smote, Y_full_train_smote)
    print("✓ Final model retrained dengan data SMOTE")

    # ==========================================
    # 8. EVALUASI FINAL DI TEST SET
    #    Menggunakan final_model + threshold kalibrasi
    # ==========================================
    print("\n🏆 8. Evaluasi Final di TEST SET...")
    print("   (Final model SMOTE + Threshold dari kalibrasi data asli)")

    y_test_proba = chain_predict_proba_threshold(
        final_model,      # ← model utama dari pickle (SMOTE)
        X_test,
        best_thresholds   # ← threshold dari kalibrasi data asli
    )

    y_test_optimized = np.zeros((len(X_test), len(target_cols)))
    for i in range(len(target_cols)):
        y_prob                 = y_test_proba[i][:, 1]
        y_test_optimized[:, i] = (y_prob >= best_thresholds[i]).astype(int)

    # ==========================================
    # 9. KALKULASI METRIK
    # ==========================================
    print("\n📊 9. Menghitung Metrik Evaluasi...")

    macro_f1    = f1_score(Y_test, y_test_optimized, average='macro',  zero_division=0)
    micro_f1    = f1_score(Y_test, y_test_optimized, average='micro',  zero_division=0)
    hamming     = hamming_loss(Y_test, y_test_optimized)
    exact_match = accuracy_score(Y_test, y_test_optimized)

    per_label_f1_scores = {}
    for i, target_name in enumerate(target_cols):
        f1 = f1_score(Y_test.iloc[:, i], y_test_optimized[:, i], zero_division=0)
        per_label_f1_scores[target_name] = f1

    print("-" * 60)
    print(f"🌟 FINAL MACRO F1-SCORE : {macro_f1:.4f}")
    print(f"🌟 FINAL MICRO F1-SCORE : {micro_f1:.4f}")
    print(f"   Hamming Loss         : {hamming:.4f}")
    print(f"   Exact Match Accuracy : {exact_match:.4f}")
    print("-" * 60)

    print("\nPer-Label F1 Scores:")
    for label, f1_val in per_label_f1_scores.items():
        print(f"   → {label}: {f1_val:.4f}")

    # ==========================================
    # 10. CLASSIFICATION REPORT
    # ==========================================
    print("\n📋 Classification Report:")
    class_report = classification_report(
        Y_test, y_test_optimized,
        target_names=['Depression', 'Anxiety', 'Stress'],
        zero_division=0
    )
    print(class_report)

    # ==========================================
    # 11. MLflow LOGGING
    # ==========================================
    print("\n📈 11. Logging ke MLflow...")

    mlflow.log_param("strategy",               "Threshold Optimization - Kalibrasi Data Asli")
    mlflow.log_param("val_split_ratio",        0.2)
    mlflow.log_param("num_features",           len(selected_features))
    mlflow.log_param("num_labels",             len(target_cols))
    mlflow.log_param("threshold_search_range", "0.1-0.9")
    mlflow.log_param("threshold_search_step",  0.01)
    mlflow.log_param("threshold_source",       "Data Asli (tanpa SMOTE)")
    mlflow.log_param("final_model_source",     "Pickle (dilatih dengan SMOTE)")
    for k, v in best_params.items():
        mlflow.log_param(f"xgb_{k}", v)

    mlflow.log_metric("test_macro_f1",             macro_f1)
    mlflow.log_metric("test_micro_f1",             micro_f1)
    mlflow.log_metric("test_hamming_loss",         hamming)
    mlflow.log_metric("test_exact_match_accuracy", exact_match)
    for label, f1_val in per_label_f1_scores.items():
        mlflow.log_metric(f"test_f1_{label}", f1_val)
    mlflow.log_metric("train_sub_samples",  X_train_sub.shape[0])
    mlflow.log_metric("validation_samples", X_val.shape[0])
    mlflow.log_metric("test_samples",       X_test.shape[0])

    report_path = "classification_report.txt"
    with open(report_path, "w") as f:
        f.write(class_report)
    mlflow.log_artifact(report_path)

    thresholds_path = "optimal_thresholds.txt"
    with open(thresholds_path, "w") as f:
        f.write("Optimal Thresholds per Label:\n")
        f.write("-" * 40 + "\n")
        for label, threshold in zip(target_cols, best_thresholds):
            f.write(f"{label}: {threshold:.4f}\n")
    mlflow.log_artifact(thresholds_path)
    print("✓ MLflow logging complete")

    # ==========================================
    # 12. SUMMARY
    # ==========================================
    print("\n" + "="*70)
    print("🎉 THRESHOLD OPTIMIZATION SELESAI")
    print("="*70)
    print(f"Strategy  : Final Model SMOTE + Threshold Kalibrasi Data Asli")
    print(f"\nValidation F1 (data asli):")
    for label, f1_val in validation_f1_scores.items():
        print(f"   {label}: {f1_val:.4f}")
    print(f"\nTest Set Results:")
    print(f"   Macro F1-Score : {macro_f1:.4f}")
    print(f"   Micro F1-Score : {micro_f1:.4f}")
    print(f"   Hamming Loss   : {hamming:.4f}")
    print(f"   Exact Match    : {exact_match:.4f}")
    print(f"\nOptimal Thresholds (dari kalibrasi data asli):")
    for label, threshold in zip(target_cols, best_thresholds):
        print(f"   {label}: {threshold:.4f}")
    print("="*70)

d:\Amikom\Semester 6\Project Data Mining\Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ 1. Memuat Data...
✓ Train SMOTE  : (8724, 31)
✓ Train Asli   : (21740, 31)
✓ Test         : (5436, 31)

🔀 2. Split Data Asli → Train_sub & Validation (80-20)
✓ Train sub (asli) : (17392, 31)
✓ Validation (asli): (4348, 31)

Distribusi Validation (asli):
risk_depression    0.7150
risk_anxiety       0.7364
risk_stress        0.6118
dtype: float64

Distribusi Test:
risk_depression    0.7068
risk_anxiety       0.7349
risk_stress        0.6078
dtype: float64

📦 3. Load Final Model dari Pickle...
✓ Final model loaded: ClassifierChain
✓ Best params dari pickle:
   - n_estimators: 100
   - max_depth: 3
   - learning_rate: 0.2
   - subsample: 0.7
   - colsample_bytree: 1.0
   - gamma: 0.5
   - min_child_weight: 5

✓ Helper functions defined

🔧 5. Training model sementara di data asli (untuk kalibrasi threshold)...
✓ Model sementara selesai dilatih
  (Model ini hanya untuk kalibrasi threshold, akan dibuang setelahnya)

🔍 6. Mencari Threshold Optimal di Validation ASLI...

   Target            

In [2]:
# Prediksi dengan threshold default 0.5
y_pred_default = final_model.predict(X_test)

macro_f1_default = f1_score(Y_test, y_pred_default, average='macro', zero_division=0)
hamming_default  = hamming_loss(Y_test, y_pred_default)
exact_default    = accuracy_score(Y_test, y_pred_default)

print("Perbandingan:")
print(f"{'Metrik':<25} {'Default(0.5)':>12} {'SMOTE Threshold':>16} {'Asli Threshold':>15}")
print("-"*70)
print(f"{'Macro F1':<25} {macro_f1_default:>12.4f} {0.8224:>16.4f} {0.7936:>15.4f}")
print(f"{'Hamming Loss':<25} {hamming_default:>12.4f} {0.2425:>16.4f} {0.2661:>15.4f}")
print(f"{'Exact Match':<25} {exact_default:>12.4f} {0.5754:>16.4f} {0.5620:>15.4f}")

Perbandingan:
Metrik                    Default(0.5)  SMOTE Threshold  Asli Threshold
----------------------------------------------------------------------
Macro F1                        0.7799           0.8224          0.7936
Hamming Loss                    0.2768           0.2425          0.2661
Exact Match                     0.5561           0.5754          0.5620
